In [1]:
!pip install sentence-transformers transformers faiss-cpu pypdf numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 34.2 MB/s eta 0:00:00


In [2]:
from google.colab import files
uploaded = files.upload()

Saving atomic structure.pdf to atomic structure.pdf


In [3]:
from pypdf import PdfReader

file_name = list(uploaded.keys())[0]

reader = PdfReader(file_name)

text = ""
for page in reader.pages:
    text += page.extract_text()

In [4]:
CONFIG = {
    "chunk_size": 500,
    "top_k": 3,
    "model_name": "all-MiniLM-L6-v2"
}

In [5]:
chunks = []
metadata = []

for i in range(0, len(text), CONFIG["chunk_size"]):
    chunk = text[i:i+CONFIG["chunk_size"]]
    chunks.append(chunk)
    metadata.append({
        "chunk_id": i,
        "length": len(chunk)
    })

print("Total chunks:", len(chunks))

Total chunks: 237


In [6]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer(CONFIG["model_name"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
embedding_cache = {}

def get_embedding(text):
    if text in embedding_cache:
        return embedding_cache[text]

    emb = model.encode([text])[0]
    embedding_cache[text] = emb
    return emb

embeddings = np.array([get_embedding(c) for c in chunks])

In [8]:
import faiss

dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)

index.add(embeddings)

In [9]:
import faiss

dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)

index.add(embeddings)

In [10]:
from transformers import pipeline

qa_pipeline = pipeline(
    "question-answering",
    model="distilbert-base-uncased-distilled-squad"
)

config.json:   0%|          | 0.00/451 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/265M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [11]:
chat_history = []

def build_query(query):
    history_text = " ".join(chat_history[-3:])
    return history_text + " " + query

In [12]:
def retrieve(query, top_k=3):
    query_vec = model.encode([query])

    D, I = index.search(np.array(query_vec), k=top_k * 2)

    candidates = [chunks[i] for i in I[0]]

    # Re-ranking (simple heuristic)
    ranked = sorted(candidates, key=lambda x: len(x), reverse=True)

    return ranked[:top_k]

In [13]:
import time

def answer_query(query):
    start = time.time()

    enhanced_query = build_query(query)

    retrieved = retrieve(enhanced_query, CONFIG["top_k"])
    context = " ".join(retrieved)

    result = qa_pipeline(question=query, context=context)

    end = time.time()

    chat_history.append(query)

    return {
        "answer": result["answer"],
        "confidence": result["score"],
        "response_time": round(end - start, 2),
        "context": retrieved
    }

In [14]:
import logging

logging.basicConfig(filename="rag.log", level=logging.INFO)

def log_query(query, result):
    logging.info(f"Query: {query}")
    logging.info(f"Answer: {result['answer']}")
    logging.info(f"Time: {result['response_time']} sec\n")

In [15]:
query = input("Ask a question: ")

result = answer_query(query)

log_query(query, result)

print("\nAnswer:", result["answer"])
print("Confidence:", result["confidence"])
print("Response Time:", result["response_time"], "sec")

print("\n--- Retrieved Chunks ---\n")
for i, chunk in enumerate(result["context"]):
    print(f"Chunk {i+1}:\n{chunk[:200]}\n")

Ask a question: atoms are made up of

Answer: molecules or ions which have the same number of electrons
Confidence: 0.04513607546687126
Response Time: 0.15 sec

--- Retrieved Chunks ---

Chunk 1:
 on law of mass conservation and law of definite proportions.  The salient feature's of
this theory are :-
(1) Each element is composed of extremely small particles called atoms.
(2) Atoms of a partic

Chunk 2:
 write the electronic configurations of atoms.
"Imagination is more important than knowledge. "
Albert EinsteinZ:\NODE02\B0AI-B0\TARGET\CHEM\ENG\MODULE-1\2.ATOMIC STRUCTURE\01-THEORY.P65
41E
Pre-Medic

Chunk 3:
osters
They are the molecules which have the same number of atoms & electrons.
Ex.1 CO2 N2O Ex.2 CaO KF
Atoms = 1 + 2 Atoms    = 2 + 1 Atoms     = 2 A t o m s =2
= 3     = 3 Electrons  = 20 + 8 Electr



In [16]:
def highlight(context, answer):
    return context.replace(answer, f"🔴{answer}🔴")

print("\n--- Highlighted Context ---\n")

for chunk in result["context"]:
    print(highlight(chunk, result["answer"])[:300])


--- Highlighted Context ---

 on law of mass conservation and law of definite proportions.  The salient feature's of
this theory are :-
(1) Each element is composed of extremely small particles called atoms.
(2) Atoms of a particular element are like but differ from atoms of other element.
(3) Atom of each element is an ultimat
 write the electronic configurations of atoms.
"Imagination is more important than knowledge. "
Albert EinsteinZ:\NODE02\B0AI-B0\TARGET\CHEM\ENG\MODULE-1\2.ATOMIC STRUCTURE\01-THEORY.P65
41E
Pre-Medical  : Chem
istryALLEN
2.0 IN TROD
UCTION
Atom is a Greek word and its  meaning is indivisible i.e. a
osters
They are the molecules which have the same number of atoms & electrons.
Ex.1 CO2 N2O Ex.2 CaO KF
Atoms = 1 + 2 Atoms    = 2 + 1 Atoms     = 2 A t o m s =2
= 3     = 3 Electrons  = 20 + 8 Electrons = 19 + 9
Electrons = 6 + 8 ×  2 Electrons = 7 ×  2 + 8       = 28 e = 28 e
= 22 e      = 22e
(f)
